In [1]:
from IPython.display import display
from datetime import datetime
from time import sleep

import sys
import os
import requests
import re

import pandas as pd

sys.path.append(os.path.abspath("./"))
import helpers as _

In [7]:
# http://api.steampowered.com/ISteamApps/GetAppList/v0002/?key=STEAMKEY&format=json

#df = pd.read_json('Steam/data/game_ids.json')
#df['name'] = df['name'].str.strip()

#df = df[
#    ~df['name'].eq("")
#    & ~df['name'].str.startswith('test', na=False) 
#    & ~df['name'].str.contains('(?i)demo| test ', na=False) 
#]
#df['checked'] = False
#df = df.rename(columns={'appid': 'id'})

#df.to_csv('Steam/data/game_ids.csv', index=False)

#df

,id,name,checked
0,2240760,Fantasy Grounds - Pathfinder RPG - Pathfinder ...,True
1,2240780,307 Racing,True
2,2240790,Sucker for Love: Date to Die For,True
3,2240810,Fantasy Grounds - Curse of Ra'khan,True
4,2240820,Fantasy Grounds - Terrible Beauty,True
...,...,...,...
197958,2908120,One Boss One Fight,False
197959,2774800,FallNation Lost Stories,False
197960,2562100,Deadlocked,False
197961,3309620,Lavender Dream,False


In [2]:
BASE_DIR = 'Steam/data/'

def prepare_headers(content_type='application/json'):
    return {
        "Content-Type": content_type
    }

def call_api_steam_get(method, params=dict(), content_type="application/json"):
    API_BASE_URI = "http://store.steampowered.com/api/"

    response = requests.get(API_BASE_URI + method, params=params, headers=prepare_headers(content_type))
                
    print(f'Response : {response.status_code}')
    
    return response

def call_api_game_details(app_id, try_nb=0):
    app_id = str(app_id)

    # https://store.steampowered.com/api/appdetails?appids=2240900
    response = call_api_steam_get('appdetails', {'appids':app_id})

    if response.status_code == 429:
        if try_nb < 10:
            print('/!\ Too much try')
            sleep(120)
            return call_api_game_details(app_id, try_nb+1)
        else:
            os.exit('Too much try')
    
    if response.status_code == 200:
        json = response.json()

        if app_id in json:
            details = json[str(app_id)]

            if details['success']:
                data = details['data']
                
                #print(list(data.keys()))

                platforms = data['platforms']
                app_type = data['type']
                is_fullgame = 'fullgame' not in data

                #print(f"Windows : {platforms['windows']}")
                #print(f"Fullgame : {is_fullgame}")
                #print(f"Type : {app_type}")

                if platforms['windows'] and is_fullgame and app_type == 'game' :
                    languages = None
                    categories = None
                    genres = None
                    
                    if 'supported_languages' in data:
                        languages = re.sub('<[^<]+?>', '', data['supported_languages'])\
                            .replace('*', '')\
                            .replace('languages with full audio support', '')\
                            .replace('Langues avec support audio complet', '')\
                            .split(',')
                        languages = '|'.join([l.strip() for l in languages])

                    if 'categories' in data:
                        categories = '|'.join([c['description'] for c in data['categories']]) # [{'id': 2, 'description': 'Single-player'}]
                        
                    if 'genres' in data:
                        genres = '|'.join([g['description'] for g in data['genres']]) # [{'id': '23', 'description': 'Indie'}]

                    # 'currency': 'EUR', 'initial': 2450, 'final': 1960, 'discount_percent': 20, 'initial_formatted': '24,50€', 'final_formatted': '19,60€'
                    # image_header = https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/{id}/header.jpg
                    
                    return pd.DataFrame([{
                        'id': data['steam_appid'],
                        'name': data['name'],
                        'categories': categories,
                        'genres': genres,
                        'languages': languages,
                        'price_amount': data['price_overview']['initial'] / 100 if 'price_overview' in data else None,
                        'price_currency': data['price_overview']['currency'] if 'price_overview' in data else None,
                        'developers': '|'.join(data['developers']) if 'developers' in data else None,
                        'date_release': data['release_date']['date'] if not data['release_date']['coming_soon'] else None,
                        'ts': datetime.now()
                    }])

    return pd.DataFrame()
    

def chunk_list(lst, chunk_size):
    for i in range(0, len(lst), chunk_size):
        yield lst[i:i + chunk_size]

In [3]:
#df_games = pd.read_csv('Steam/data/games.csv')

#df_games.to_csv('Steam/data/games.csv', index=False)  

In [5]:
df_ids = pd.read_csv('Steam/data/game_ids.csv')
df_games = pd.read_csv('Steam/data/games.csv')

for ids in chunk_list(df_ids[~df_ids['checked']]['id'].values, 100):
    for index, id in enumerate(ids):
        print('')
        print(f'Query : {index} {id}')
    
        df_game = call_api_game_details(id)

        if df_game.shape[0] > 0 and df_games[df_games['id'] == df_game['id'].values[0]].shape[0] == 0:
            print(f'Added : {id}')
            df_games = pd.concat([df_games, df_game], ignore_index=True)

        df_ids.loc[df_ids['id'] == id,'checked'] = True
    
    df_games.to_csv('Steam/data/games.csv', index=False)  
    df_ids.to_csv('Steam/data/game_ids.csv', index=False)    
    
    sleep(120)

print('End')


Query : 0 2197670
Response : 200
Added : 2197670

Query : 1 2196810
Response : 200
Added : 2196810

Query : 2 2196820
Response : 200
Added : 2196820

Query : 3 2196840
Response : 200
Added : 2196840

Query : 4 2196850
Response : 200
Added : 2196850

Query : 5 2196870
Response : 200
Added : 2196870

Query : 6 2196880
Response : 200

Query : 7 2196881
Response : 200

Query : 8 2196890
Response : 200

Query : 9 2196910
Response : 200
Added : 2196910

Query : 10 2196920
Response : 200
Added : 2196920

Query : 11 2196930
Response : 200
Added : 2196930

Query : 12 2196940
Response : 200
Added : 2196940

Query : 13 2196950
Response : 200
Added : 2196950

Query : 14 2196970
Response : 200

Query : 15 2196980
Response : 200
Added : 2196980

Query : 16 2197000
Response : 200
Added : 2197000

Query : 17 2197020
Response : 200
Added : 2197020

Query : 18 2197030
Response : 200
Added : 2197030

Query : 19 2197050
Response : 200
Added : 2197050

Query : 20 2197060
Response : 200
Added : 2197060

Qu

JSONDecodeError: Expecting value: line 1 column 1 (char 0)